# Experiment 2: Data Preprocessing, Pipelines and Leakage Detection

**Dataset:** `Titanic-Dataset.xls` (CSV-formatted text)  
**Target:** `Survived`  
**Objective:** Compare flawed preprocessing with a leakage-free scikit-learn pipeline.

## 1. Predict: where leakage can occur

Test information can leak into training whenever a data-dependent operation is fitted before the train/test split. Likely leakage points include computing imputation values from all passengers, computing scaling means and standard deviations from all passengers, discovering encoding categories from the full dataset, selecting features using the target before splitting, or directly retaining a target-derived column among the features. All learned preprocessing must therefore be fitted only on the training partition.

## 2. Setup and feature selection

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 67

DATA_PATH = Path("Titanic-Dataset.xls")
df = pd.read_csv(DATA_PATH)

numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]
feature_columns = numeric_columns + categorical_columns

X = df[feature_columns].copy()
y = df["Survived"].copy()

print("Selected features:", feature_columns)
print("Feature matrix:", X.shape)
print("Target distribution:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

Selected features: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex', 'Embarked']
Feature matrix: (891, 7)
Target distribution:


,proportion
Survived,
0,0.616162
1,0.383838


`PassengerId`, `Name`, `Ticket`, and `Cabin` are excluded here to keep the introductory pipeline interpretable and avoid high-cardinality identifiers. This is a modelling choice, not a preprocessing result. The target is separated before any transformation.

## 3. Inspect the flawed pipeline

In [2]:
def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ])


# FLAWED: the preprocessor learns from the complete dataset, including future test rows.
leaky_preprocessor = make_preprocessor()
X_processed_full = leaky_preprocessor.fit_transform(X)

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_processed_full,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

leaky_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
leaky_model.fit(X_train_leaky, y_train_leaky)
leaky_predictions = leaky_model.predict(X_test_leaky)
print("Flawed order: fit preprocessing on all rows -> split -> fit model")

Flawed order: fit preprocessing on all rows -> split -> fit model


**Bug annotation:** `fit_transform(X)` computes imputers, scaling statistics, and encoding categories using both training and test rows. The labels are not used, but the model-development process has still looked at the held-out feature distribution, so the test set is no longer a fully independent simulation of unseen data.

## 4. Rebuild the workflow correctly: split first

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

corrected_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

# Pipeline.fit learns every preprocessing statistic only from X_train.
corrected_pipeline.fit(X_train, y_train)
corrected_predictions = corrected_pipeline.predict(X_test)

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
print("Correct order: split -> fit preprocessing and model on training rows only")

Correct order: split -> fit preprocessing and model on training rows only


## 5. Observe: compare held-out metrics

In [4]:
def metric_row(name, truth, predictions):
    return {
        "workflow": name,
        "accuracy": accuracy_score(truth, predictions),
        "f1_score": f1_score(truth, predictions),
    }


comparison = pd.DataFrame([
    metric_row("Leaky: full preprocessing before split", y_test_leaky, leaky_predictions),
    metric_row("Corrected: split before pipeline.fit", y_test, corrected_predictions),
])
display(comparison.assign(
    accuracy=comparison["accuracy"].map("{:.2%}".format),
    f1_score=comparison["f1_score"].map("{:.2%}".format),
))

,workflow,accuracy,f1_score
0,Leaky: full preprocessing before split,80.60%,73.74%
1,Corrected: split before pipeline.fit,80.60%,73.74%


In [5]:
full_age_mean = X["Age"].mean()
train_age_mean = X_train["Age"].mean()
full_fare_mean = X["Fare"].mean()
train_fare_mean = X_train["Fare"].mean()

learned_statistics = pd.DataFrame({
    "statistic": ["Age mean", "Fare mean"],
    "leaky_full_data_value": [full_age_mean, full_fare_mean],
    "correct_training_only_value": [train_age_mean, train_fare_mean],
})
learned_statistics["difference"] = (
    learned_statistics["leaky_full_data_value"]
    - learned_statistics["correct_training_only_value"]
)
display(learned_statistics.round(4))

,statistic,leaky_full_data_value,correct_training_only_value,difference
0,Age mean,29.6991,29.8468,-0.1477
1,Fare mean,32.2042,30.7527,1.4515


## 6. Explain

The leaky score is untrustworthy because preprocessing parameters were influenced by the held-out passengers. Leakage does **not** guarantee a higher score in every finite sample: the rounded accuracy may be equal, higher, or lower. The defect is that the estimate no longer represents a clean evaluation on unseen data, so it can be optimistically biased and may fail to reproduce in deployment. The corrected pipeline preserves the test set exclusively for final transformation and evaluation.

## 7. Modify: introduce a second leakage source (full-data imputation)

In [6]:
# SECOND FLAW: fill missing values using statistics from the complete dataset.
X_imputed_full = X.copy()
full_imputation_values = {
    "Age": X["Age"].median(),
    "Fare": X["Fare"].median(),
    "Embarked": X["Embarked"].mode().iloc[0],
}
X_imputed_full = X_imputed_full.fillna(full_imputation_values)

X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imputed_full,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

after_imputation_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
])
imputation_leak_pipeline = Pipeline([
    ("preprocessor", after_imputation_preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
imputation_leak_pipeline.fit(X_train_imp, y_train_imp)
imputation_leak_predictions = imputation_leak_pipeline.predict(X_test_imp)

train_only_imputation_values = {
    "Age": X_train["Age"].median(),
    "Fare": X_train["Fare"].median(),
    "Embarked": X_train["Embarked"].mode().iloc[0],
}
display(pd.DataFrame({
    "full_data_value (leaky)": full_imputation_values,
    "training_only_value (correct)": train_only_imputation_values,
}))

,full_data_value (leaky),training_only_value (correct)
Age,28.0,28.0
Fare,14.4542,14.4542
Embarked,S,S


## 8. Perform severe target leakage

A more serious leakage source is created below by calculating every ticket group's mean `Survived` value **before** the split. This is intentionally wrong: `TicketSurvivalRate_LEAK` contains target information from the complete dataset, including the future test rows and each passenger's own outcome. The preprocessing pipeline that follows cannot undo leakage already embedded in a feature.

In [7]:
# SEVERE FLAW: engineer a feature from all target labels before splitting.
target_leak_df = df.copy()
target_leak_df["TicketSurvivalRate_LEAK"] = (
    target_leak_df.groupby("Ticket")["Survived"].transform("mean")
)

target_leak_numeric_columns = numeric_columns + ["TicketSurvivalRate_LEAK"]
X_target_leak = target_leak_df[
    target_leak_numeric_columns + categorical_columns
]

X_train_target_leak, X_test_target_leak, y_train_target_leak, y_test_target_leak = (
    train_test_split(
        X_target_leak,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )
)

target_leak_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), target_leak_numeric_columns),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_columns),
])

target_leak_pipeline = Pipeline([
    ("preprocessor", target_leak_preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
target_leak_pipeline.fit(X_train_target_leak, y_train_target_leak)
target_leak_predictions = target_leak_pipeline.predict(X_test_target_leak)

ticket_group_size = target_leak_df.groupby("Ticket")["PassengerId"].transform("size")
singleton_ticket_passengers = int(ticket_group_size.eq(1).sum())

print("Passengers whose ticket occurs once:", singleton_ticket_passengers)
print(
    "For these passengers, TicketSurvivalRate_LEAK is exactly their own target:",
    bool((
        target_leak_df.loc[ticket_group_size.eq(1), "TicketSurvivalRate_LEAK"]
        == y.loc[ticket_group_size.eq(1)]
    ).all()),
)

Passengers whose ticket occurs once: 547
For these passengers, TicketSurvivalRate_LEAK is exactly their own target: True


Because many ticket numbers occur only once, their ticket survival rate is literally the passenger's own `Survived` label. Even for shared tickets, the value contains the passenger's outcome and may contain outcomes from test passengers. This creates an unrealistically strong predictor and should produce a dramatically inflated test score.

In [8]:
all_results = pd.DataFrame([
    metric_row("Leak 1: all preprocessing before split", y_test_leaky, leaky_predictions),
    metric_row("Leak 2: full-data imputation before split", y_test_imp, imputation_leak_predictions),
    metric_row(
        "Leak 3: full-data ticket target encoding",
        y_test_target_leak,
        target_leak_predictions,
    ),
    metric_row("Corrected leakage-free pipeline", y_test, corrected_predictions),
])
display(all_results.assign(
    accuracy=all_results["accuracy"].map("{:.2%}".format),
    f1_score=all_results["f1_score"].map("{:.2%}".format),
))

,workflow,accuracy,f1_score
0,Leak 1: all preprocessing before split,80.60%,73.74%
1,Leak 2: full-data imputation before split,80.60%,73.74%
2,Leak 3: full-data ticket target encoding,98.51%,98.10%
3,Corrected leakage-free pipeline,80.60%,73.74%


## 9. Leakage report

| Leakage point | Why it is wrong | Correction |
|---|---|---|
| Scaling before splitting | Test-set means and standard deviations influence training-time feature values. | Split raw rows first; place `StandardScaler` inside a pipeline fitted on training data. |
| Encoding before splitting | Categories are discovered using held-out rows. | Fit `OneHotEncoder(handle_unknown="ignore")` only through the training pipeline. |
| Imputation before splitting | Full-data medians/modes contain information about the held-out feature distribution. | Place `SimpleImputer` inside the training-fitted pipeline. |
| Ticket target encoding before splitting | `TicketSurvivalRate_LEAK` is calculated using all `Survived` labels, including test outcomes and each row's own outcome. | Do not use it for this model; when target encoding is justified, generate training values out-of-fold and map test values using training labels only. |
| Test set used during development | Repeated choices based on test scores indirectly overfit the test set. | Develop with training/validation or cross-validation; use the test set once for final evaluation. |

## 10. Reflect and conclude

**General rule:** split the raw observations before performing any operation that learns from data. Put imputation, encoding, scaling, feature selection, and the estimator in one pipeline; fit that pipeline only on training data. During model development, use cross-validation whose preprocessing is fitted independently inside each fold, and reserve the untouched test set for one final evaluation. Never use the target—or information created after prediction time—as an input feature. If target encoding is needed, construct training encodings out-of-fold and derive validation/test mappings exclusively from training labels.